In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

### Change in the phase transition with respect to the decay parameter λ

In [3]:
import os
from pathlib import Path
from mintrans_clean import *



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

l = [0.9, 0.92, 0.95, 0.98, 0.99, 1.0]

checkpoint_dir = './skeletons_var_lanbda'
if not os.path.exists(checkpoint_dir): 
    os.makedirs(checkpoint_dir)
# general_paramater in path refers to the configuration in minitrans_clean.py with different weight decay



# ---------------------------------------------------------------------------
# General configuration
# ---------------------------------------------------------------------------

cfg = TrainConfig

# ---------------------------------------------------------------------------
# Data loader configuration
# ---------------------------------------------------------------------------

torch.manual_seed(cfg.torch_seed)

train_ds = FibonacciTrainDataset(
    mod=cfg.p, seq_len=cfg.seq_len, train_frac=cfg.train_frac,
    seed=cfg.data_seed,
)
val_ds = FibonacciValDataset(train_ds, num_samples=500)

n_unseen = sum(
    1 for a in range(cfg.p) for b in range(cfg.p)
    if (a, b) not in train_ds.seen_pairs
)

cfg.batch_size = len(train_ds)   
cfg.epochs = 8000

train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size,
    num_workers=4, pin_memory=True, persistent_workers=True,
)


for i in range(len(l)):

    # ---------------------------------------------------------------------------
    # Model configuration 
    # ---------------------------------------------------------------------------

    model = MinimalTransformer(
        vocab_size=cfg.vocab_size, d_model=cfg.d_model, n_heads=cfg.n_heads,
        num_layers=cfg.num_layers, max_seq_len=cfg.seq_len,
        hidden_mlp=cfg.hidden_mlp,
    ).to(device)

    weight_decay = l[i]

    cfg.weight_decay = weight_decay
    cfg.checkpoint_path = os.path.join(checkpoint_dir, f'general_paramater_wd_{weight_decay}.pth')

    try:
        history = train_model(model, train_loader, val_loader, cfg)
    except KeyboardInterrupt:
        history = {k: [] for k in ("train_loss", "val_loss", "train_acc", "val_acc", "spectral")}

    val_loss, val_acc     = evaluate(model, val_loader)
    train_loss, train_acc = evaluate(model, train_loader)

    print(f"\nFinal — train acc: {train_acc:.3f}  val acc: {val_acc:.3f}")

    if cfg.checkpoint_path:
        save_checkpoint(model, history, cfg.checkpoint_path)



Epoch   100/8000  train loss 2.2179 acc 0.493  val loss 7.7291 acc 0.032
Epoch   200/8000  train loss 0.0187 acc 1.000  val loss 15.4990 acc 0.098
Epoch   300/8000  train loss 0.0066 acc 1.000  val loss 15.8264 acc 0.118
Epoch   400/8000  train loss 0.0021 acc 1.000  val loss 16.5499 acc 0.116
Epoch   500/8000  train loss 0.0007 acc 1.000  val loss 17.3335 acc 0.128
Epoch   600/8000  train loss 0.0002 acc 1.000  val loss 18.1625 acc 0.134
Epoch   700/8000  train loss 0.0001 acc 1.000  val loss 19.0179 acc 0.130
Epoch   800/8000  train loss 0.0000 acc 1.000  val loss 19.8786 acc 0.134
Epoch   900/8000  train loss 0.0000 acc 1.000  val loss 20.6856 acc 0.140
Epoch  1000/8000  train loss 0.0000 acc 1.000  val loss 21.3978 acc 0.148
Epoch  1100/8000  train loss 0.0000 acc 1.000  val loss 21.9839 acc 0.152
Epoch  1200/8000  train loss 0.0000 acc 1.000  val loss 22.3777 acc 0.156
Epoch  1300/8000  train loss 0.0000 acc 1.000  val loss 22.5161 acc 0.158
Epoch  1400/8000  train loss 0.0000 acc